# Fase 2 — Otimização dos modelos

Reproduz os três experimentos genéticos para cada um dos cinco classificadores nos estudos mama e PCOS. Reutiliza o código da aplicação, sem duplicar o algoritmo. Não requer API nem chave Gemini. O processamento pode levar vários minutos.

Genes discretos, seleção por torneio, crossover uniforme, mutação por gene e elitismo. Split estratificado 80/20; CV estratificada de três folds somente no treino; fitness = 0,50 recall + 0,30 F1 + 0,20 acurácia. O holdout é reservado para comparar baseline e parâmetros selecionados. Ganhos não são garantidos nem validação clínica.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src' / 'optimization' / 'experiments.py').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from src.optimization.cli import main

assert main(['--study', 'all']) == 0


In [ ]:
import json
import pandas as pd
from IPython.display import display

RESULTS_PATH = PROJECT_ROOT / 'src' / 'db' / 'outputs' / 'phase2-genetic' / 'optimization_results.json'
results = json.loads(RESULTS_PATH.read_text(encoding='utf-8'))
rows = []
for study, data in results['studies'].items():
    assert len(data['models']) == 5
    for model, result in data['models'].items():
        assert len(result['experiments']) == 3
        rows.append({'study': study, 'model': model, 'experiment': result['optimized']['selected_experiment'], 'recall_baseline': result['baseline']['test_metrics']['recall'], 'recall_GA': result['optimized']['test_metrics']['recall'], 'F1_baseline': result['baseline']['test_metrics']['f1'], 'F1_GA': result['optimized']['test_metrics']['f1']})
display(pd.DataFrame(rows))
print(f'Resultados persistidos: {RESULTS_PATH}')


## Interpretação Gemini opcional

A tabela acima é calculada localmente. Para consultar a LLM, inicie `.\run.ps1` em outro terminal e informe sua chave com entrada oculta. Depois execute esta célula no Jupyter. Não há chamadas remotas na execução batch; esta célula é marcada como `interactive`.

A interface dos exports de mama e PCOS continua disponível. O notebook seguinte verifica os contratos da API, interpretação local, cache e infraestrutura opcional.

In [ ]:
from src.optimization.notebook_ui import display_phase2_interpretation

clinical_ui = display_phase2_interpretation('breast_cancer')
